# California Real Estate — Exploratory Data Analysis

**Data Source:** CRMLS (California Regional Multiple Listing Service) via CoreLogic Trestle API  
**Coverage:** January 2024 — March 2026 (27 months)  
**Datasets:**
- `priceratio.csv` — Closed residential transactions with derived market metrics
- `newlistings.csv` — New residential listings

**Objective:** Identify meaningful patterns and relationships in the California housing market to inform the construction of market analysis and competitive analysis Tableau dashboards.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Add project root to path so we can import our modules
sys.path.insert(0, os.path.abspath('..'))

from data_cleaning.helpers.duplicates import drop_duplicate_columns
from data_cleaning.helpers.dates import parse_sold_dates, parse_listing_dates
from data_cleaning.helpers.missing import missing_value_report
from data_cleaning.helpers.outliers import flag_outliers, filter_outliers

from feature_engineering.helpers.engineer_features import (
    add_market_condition, add_price_reduction_flags,
    add_dom_buckets, add_price_tiers
)
from feature_engineering.add_new_features import engineer_sold_features, engineer_listing_features

from exploratory_analysis.helpers.visualize import (
    plot_missing_values, plot_boxplots, plot_sold_distributions,
    plot_listing_distributions, plot_monthly_kpi_trends, plot_yoy_comparison,
    plot_supply_vs_demand, plot_top_counties, plot_top_cities,
    plot_subtype_comparison, plot_price_vs_size, plot_dom_bucket_analysis,
    plot_price_reductions, plot_top_offices, plot_top_agents,
    plot_correlation_heatmap, plot_close_vs_list, plot_price_tiers,
    plot_new_construction
)
from exploratory_analysis.helpers.stats import (
    monthly_sold_summary, monthly_listing_summary, geographic_summary,
    dom_bucket_summary, competitive_summary, price_tier_summary, market_summary
)

---
## 1. Load Data


In [ ]:
sold = pd.read_csv('priceratio.csv', encoding='ISO-8859-1')
listings = pd.read_csv('newlistings.csv', encoding='ISO-8859-1')

print(f"Sold transactions: {sold.shape[0]:,} rows x {sold.shape[1]} columns")
print(f"New listings:      {listings.shape[0]:,} rows x {listings.shape[1]} columns")

---
## 2. Data Overview

Inspect schema, data types, and summary statistics for both datasets.


In [ ]:
print("=== Sold Transactions (priceratio.csv) ===")
print(f"Columns: {sold.columns.tolist()}")
sold.info()

In [ ]:
sold.describe()


In [ ]:
print("=== New Listings (newlistings.csv) ===")
print(f"Columns: {listings.columns.tolist()}")
listings.info()

In [ ]:
listings.describe()

---
## 3. Data Cleaning

### 3a. Handle Duplicate Columns in Listings

The  file contains duplicate column names from the API extraction (e.g., , ). These are redundant and should be dropped before analysis.


In [ ]:
listings = drop_duplicate_columns(listings)

### 3b. Parse Date Columns and Create Time Features

In [ ]:
sold = parse_sold_dates(sold)
listings = parse_listing_dates(listings)

### 3c. Missing Data Analysis

In [ ]:
sold_missing = missing_value_report(sold, "Sold Transactions")

In [ ]:
list_missing = missing_value_report(listings, "New Listings")

In [ ]:
plot_missing_values(sold_missing, list_missing)

### 3d. Outlier Assessment

The  and  fields in the sold dataset already have IQR-based outlier handling applied (outliers replaced with NaN in ). Here we examine the remaining distributions for any additional anomalies in key numeric fields.


In [ ]:
sold_numeric_cols = ['ClosePrice', 'OriginalListPrice', 'ListPrice', 'LivingArea',
                     'DaysOnMarket', 'priceratio', 'pricesqft']
plot_boxplots(sold, sold_numeric_cols, 'Sold Data — Outlier Assessment', color='steelblue')

In [ ]:
list_numeric_cols = ['ListPrice', 'OriginalListPrice', 'LivingArea', 'DaysOnMarket',
                     'BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt']
plot_boxplots(listings, list_numeric_cols, 'Listings Data — Outlier Assessment', color='darkorange')

---
## 4. Distributions

Examine the shape of key numeric variables to understand skewness, central tendency, and spread. This informs whether we should use median vs. mean for aggregations and whether transformations are needed.


In [ ]:
plot_sold_distributions(sold)

In [ ]:
plot_listing_distributions(listings)

---
## 5. Feature Engineering

Create additional features that will be useful for both the EDA and the downstream Tableau dashboards.


In [ ]:
sold = engineer_sold_features(sold)

In [ ]:
listings = engineer_listing_features(listings)

---
## 6. Exploratory Data Analysis

### 6a. Market Trends Over Time

These monthly time-series views mirror the KPIs tracked in the CA Market Analysis Tableau dashboard: median sales price, price per square foot, days on market, and sold/list price ratio.


In [ ]:
monthly_sold = monthly_sold_summary(sold)
monthly_listings = monthly_listing_summary(listings)
plot_monthly_kpi_trends(monthly_sold, monthly_listings)

In [ ]:
plot_yoy_comparison(sold)

### 6b. Supply vs. Demand

Compare new listing volume to closed sales volume over time. The ratio of these two indicates market tightness — a rising ratio means more inventory relative to demand.


In [ ]:
plot_supply_vs_demand(monthly_sold, monthly_listings)

### 6c. Geographic Analysis

Analyze market performance at the county and city level — the primary geographic dimensions used in the Tableau market analysis dashboard.


In [ ]:
county_stats = geographic_summary(sold, 'CountyOrParish')
plot_top_counties(county_stats)

In [ ]:
city_stats = geographic_summary(sold, 'City')
plot_top_cities(city_stats)

### 6d. Property Characteristics and Pricing Relationships

Explore how property attributes (size, bedrooms, year built, property subtype) relate to price and market velocity.


In [ ]:
plot_subtype_comparison(sold)

In [ ]:
plot_price_vs_size(sold)

In [ ]:
dom_analysis = dom_bucket_summary(sold)
plot_dom_bucket_analysis(dom_analysis)

### 6e. Price Reductions Analysis

Examine the prevalence and magnitude of price reductions across markets. High reduction rates signal softening demand — a key indicator for the market analysis dashboard.


In [ ]:
plot_price_reductions(sold)

### 6f. Competitive Analysis — Agent and Office Performance

Preliminary analysis of listing agent and office performance. These metrics feed directly into the Agent/Office Report Tableau dashboard.


In [ ]:
office_stats = competitive_summary(sold, 'ListOfficeName')
plot_top_offices(office_stats)

In [ ]:
agent_stats = competitive_summary(sold, 'ListAgentFullName')
plot_top_agents(agent_stats)

### 6g. Correlation Analysis

Examine linear relationships between key numeric variables in the sold dataset to identify which features are most predictive of price, market velocity, and competitiveness.


In [ ]:
plot_correlation_heatmap(sold)

In [ ]:
plot_close_vs_list(sold)

### 6h. Price Tier Analysis

Break down key metrics by price segment to understand how market dynamics differ across price points.


In [ ]:
tier_stats = price_tier_summary(sold)
plot_price_tiers(tier_stats)

### 6i. New Construction vs. Existing Homes

In [ ]:
plot_new_construction(sold)

---
## 7. Summary Statistics for Tableau

Generate a concise summary table of the key metrics that will populate the Tableau dashboards.


In [ ]:
market_summary(sold, listings)

---
## 8. Key Findings

### Market Trends
- **Median close price is \,000** across all 27 months -- California's residential market remains firmly high-priced.
- **Average sold/list ratio of 0.9878** means the typical home sells ~1.2% below ask. However, **44.2% of transactions close at or above asking price**, indicating a split market.
- **Average DOM of 27.3 days**, with the median at 16 days. The right-skewed distribution confirms **median is the correct aggregation for DOM in Tableau**.
- Monthly KPI trends show **clear seasonality**: price peaks in late spring/summer, troughs in winter.

### Supply vs. Demand
- **539,777 new listings vs. 397,887 closed transactions**. Listing-to-sold ratio of ~1.36:1.
- The ratio trend line is a leading indicator for price direction.

### Geographic Patterns
- **63 counties and 1,123 cities** represented, confirming statewide coverage.
- Median price and DOM vary significantly by county — geographic segmentation is essential.

### Price Ratio and DOM Insights
- **DaysOnMarket and priceratio have the strongest negative correlation at -0.50** — the most actionable relationship in the dataset. Faster sales command premiums.
- Homes selling in 0-7 days have the highest average price ratio; 90+ day homes sell well below ask.

### Correlation Analysis
- **ClosePrice to pricesqft (0.84):** Validates \$/sqft as the core normalization metric.
- **BathroomsTotalInteger to ListPrice (0.52):** Bathrooms are a stronger price predictor than bedrooms.
- **LotSizeSquareFeet** has near-zero correlation with everything at the statewide level.

### Price Reductions
- **27.2% of closed sales had price reductions.** City-level rates vary from <20% to >30%.

### New Construction vs. Existing
- Existing homes: median \K at \/sqft. New construction: median \K at \/sqft.
- New construction is cheaper per sqft but sits longer on market (36.5 vs 26.9 days).

### Data Quality Notes
- FireplacesTotal and CoveredSpaces are 100% null — exclude from analysis.
- BuildingAreaTotal is 93% null — use LivingArea instead.
- LivingArea max of 17M sqft indicates data entry errors; manageable with median aggregations.
- DaysOnMarket has negative values — filter or set to 0.

### Recommended Tableau Focus Areas
1. **DOM vs. Price Ratio** — strongest finding
2. **Monthly KPI trends with YOY overlay** — clear seasonality
3. **City/county-level price ratio maps** — geographic competitiveness varies dramatically
4. **Supply vs. demand time series** — leading market indicator
5. **Price reduction rate trends** — early warning for market shifts
6. **Price tier segmentation** — dynamics differ substantially by price segment
